# ArcNeuron on Google Colab

This notebook is only a thin runner for the repository. It does not contain the architecture, reasoning rules, a knowledge base, or separate training logic.
`arcneuron.py`, `train.py`, `tune.py`, `tokenizer.py`, and `generate.py` remain the source of truth.

Enable a GPU from **Runtime → Change runtime type → GPU** before running the notebook.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO = "https://github.com/ArcatureLabs/ArcNeuron.git"
ROOT = Path("/content/ArcNeuron")

if not (ROOT / "arcneuron.py").is_file():
    if ROOT.exists():
        subprocess.run(["rm", "-rf", str(ROOT)], check=True)
    subprocess.run(["git", "clone", "--depth", "1", REPO, str(ROOT)], check=True)

os.chdir(ROOT)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentencepiece"], check=True)
print("working directory:", Path.cwd())


In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("GPU is not enabled. Select Runtime > Change runtime type > GPU and run again.")

print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("BF16:", torch.cuda.is_bf16_supported())
props = torch.cuda.get_device_properties(0)
print(f"VRAM: {props.total_memory / 1024**3:.1f} GiB")


## Experiment configuration

The defaults are intentionally small so the full pipeline can be tested quickly. Scale the model and training budget only after the pipeline works correctly.


In [ ]:
TRAIN_STEPS = 300
TUNE_STEPS = 100
BATCH_SIZE = 8
CONTEXT = 256

DIM = 256
HEADS = 4
KV_HEADS = 1
FFN_DIM = 704
PRELUDE_LAYERS = 1
CORE_LAYERS = 2
CODA_LAYERS = 1
MAX_DEPTH = 4

BASE_CKPT = "arcneuron.pt"
TUNED_CKPT = "arcneuron-tuned.pt"


## Train the base model

This cell only calls `train.py`; the notebook does not duplicate the training loop or any reasoning logic.


In [ ]:
cmd = [
    sys.executable, "train.py",
    "--data", "train.txt",
    "--out", BASE_CKPT,
    "--steps", str(TRAIN_STEPS),
    "--batch-size", str(BATCH_SIZE),
    "--context", str(CONTEXT),
    "--dim", str(DIM),
    "--heads", str(HEADS),
    "--kv-heads", str(KV_HEADS),
    "--ffn-dim", str(FFN_DIM),
    "--prelude-layers", str(PRELUDE_LAYERS),
    "--core-layers", str(CORE_LAYERS),
    "--coda-layers", str(CODA_LAYERS),
    "--max-depth", str(MAX_DEPTH),
    "--eval-every", "50",
    "--eval-batches", "4",
]
subprocess.run(cmd, check=True)


## Tuning

Tuning continues to update the same weights directly with next-token training. No adapter or auxiliary model is introduced.


In [ ]:
cmd = [
    sys.executable, "tune.py",
    "--checkpoint", BASE_CKPT,
    "--data", "tune.txt",
    "--replay-data", "train.txt",
    "--out", TUNED_CKPT,
    "--steps", str(TUNE_STEPS),
    "--batch-size", str(max(1, BATCH_SIZE // 2)),
    "--context", str(min(CONTEXT, 256)),
    "--max-depth", str(MAX_DEPTH),
    "--save-every", str(TUNE_STEPS),
]
subprocess.run(cmd, check=True)


## Generate

`DEPTH` is the number of recurrent-core applications and acts as ArcNeuron's direct test-time compute knob.


In [ ]:
from generate import load_model, generate

device = torch.device("cuda")
checkpoint = TUNED_CKPT if Path(TUNED_CKPT).is_file() else BASE_CKPT
model, tokenizer = load_model(checkpoint, device)

PROMPT = "If a cat loses one leg, is it still a mammal? Explain why."
DEPTH = 4

text = generate(
    model=model,
    tokenizer=tokenizer,
    prompt=PROMPT,
    depth=DEPTH,
    max_new_tokens=160,
    temperature=0.8,
    top_k=50,
    device=device,
)
print(text)


## Compare recurrent depth

Use the same checkpoint and prompt while changing only the recurrence count to test whether extra compute actually helps or causes overthinking.


In [ ]:
for depth in [1, 2, 4, 8]:
    torch.manual_seed(42)
    torch.cuda.manual_seed_all(42)
    text = generate(
        model=model,
        tokenizer=tokenizer,
        prompt=PROMPT,
        depth=depth,
        max_new_tokens=160,
        temperature=0.0,
        top_k=0,
        device=device,
    )
    print(f"\n{'=' * 24} depth={depth} {'=' * 24}\n")
    print(text)


## Download the checkpoint from Colab

Run this cell before the Colab runtime resets if you want to keep the checkpoint locally.


In [ ]:
from google.colab import files
path = TUNED_CKPT if Path(TUNED_CKPT).is_file() else BASE_CKPT
files.download(path)
